In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/yellow_tripdata_2024-01_engineered.parquet')
zh = pd.read_csv('../data/processed/tableau_zone_hour.csv')
zs = pd.read_csv('../data/processed/tableau_zone_summary.csv')

print("=" * 65)
print("  URBAN MOBILITY 360 - INSIGHTS REPORT")
print("  NYC Yellow Taxi - January 2024")
print("=" * 65)

# WHERE?
print("\nWHERE? - Mobility Hotspots")
print("-" * 50)
top5 = zs.nlargest(5, 'total_trips')
for _, r in top5.iterrows():
    print(f"  {r['PU_zone']} ({r['PU_borough']})")
    print(f"    -> {r['total_trips']:,.0f} trips | "
          f"Avg Speed: {r['avg_speed']:.1f} mph | "
          f"Priority: {r['avg_priority_score']:.1f}/100")

# WHEN?
print("\nWHEN? - Peak Demand Periods")
print("-" * 50)
hourly = df.groupby('pickup_hour').size().sort_values(ascending=False)
for hour, count in hourly.head(5).items():
    print(f"  {hour:02d}:00 - {count:,} trips")

wd = df[df['is_weekend']==0].shape[0]
we = df[df['is_weekend']==1].shape[0]
print(f"\n  Weekday avg daily trips: {wd//23:,}")
print(f"  Weekend avg daily trips: {we//8:,}")

# WHY?
print("\nWHY? - Congestion Drivers")
print("-" * 50)
pk = df[df['is_peak']==1]
opk = df[df['is_peak']==0]
pk_speed = pk['avg_speed_mph'].mean()
opk_speed = opk['avg_speed_mph'].mean()
drop = (1 - pk_speed/opk_speed)*100
print(f"  Peak avg speed:     {pk_speed:.1f} mph")
print(f"  Off-peak avg speed: {opk_speed:.1f} mph")
print(f"  Speed drops {drop:.1f}% during peak hours")
print(f"  Peak congestion:    {pk['is_congested_trip'].mean()*100:.1f}%")
print(f"  Off-peak congestion:{opk['is_congested_trip'].mean()*100:.1f}%")

# WHICH?
print("\nWHICH? - High-Pressure Routes")
print("-" * 50)
high_priority = zs[zs['avg_priority_score']>=35].sort_values('avg_priority_score',ascending=False)
print(f"  {len(high_priority)} zones have High+ priority scores")
for _, r in high_priority.head(5).iterrows():
    print(f"  > {r['PU_zone']} ({r['PU_borough']}) - "
          f"Score: {r['avg_priority_score']:.1f} | "
          f"Speed: {r['avg_speed']:.1f} mph | "
          f"High-hours: {r['high_hours']:.0f}")

# WHAT TO PRIORITIZE?
print("\nWHAT TO PRIORITIZE? - Recommendations")
print("-" * 50)
top_p = zs.nlargest(3, 'avg_priority_score')
for i, (_, r) in enumerate(top_p.iterrows(), 1):
    print(f"\n  {i}. {r['PU_zone']} ({r['PU_borough']})")
    if r['avg_speed'] < 11:
        print(f"     -> Low avg speed ({r['avg_speed']:.1f} mph) - consider dedicated lanes")
    if r['avg_congestion'] > 2.5:
        print(f"     -> High congestion ({r['avg_congestion']:.1f}) - signal optimization needed")
    if r['high_hours'] > 0:
        print(f"     -> {r['high_hours']:.0f} high-priority hours - extended peak management")
    if r['total_trips'] > 50000:
        print(f"     -> Very high demand ({r['total_trips']:,.0f} trips) - increase service frequency")

print("\n" + "=" * 65)
print("  REPORT COMPLETE")
print("=" * 65)

  URBAN MOBILITY 360 - INSIGHTS REPORT
  NYC Yellow Taxi - January 2024

WHERE? - Mobility Hotspots
--------------------------------------------------
  Upper East Side South (Manhattan)
    -> 130,986 trips | Avg Speed: 10.9 mph | Priority: 35.1/100
  Midtown Center (Manhattan)
    -> 125,288 trips | Avg Speed: 10.1 mph | Priority: 37.1/100
  Upper East Side North (Manhattan)
    -> 123,449 trips | Avg Speed: 11.6 mph | Priority: 33.5/100
  Lincoln Square East (Manhattan)
    -> 94,152 trips | Avg Speed: 11.4 mph | Priority: 32.8/100
  Midtown East (Manhattan)
    -> 93,826 trips | Avg Speed: 10.4 mph | Priority: 34.6/100

WHEN? - Peak Demand Periods
--------------------------------------------------
  18:00 - 175,235 trips
  17:00 - 165,543 trips
  19:00 - 149,621 trips
  15:00 - 148,253 trips
  16:00 - 148,077 trips

  Weekday avg daily trips: 75,191
  Weekend avg daily trips: 76,118

WHY? - Congestion Drivers
--------------------------------------------------
  Peak avg speed:     

In [2]:
# Save key insights as text
insights = []

top = zs.nlargest(1, 'avg_priority_score').iloc[0]
insights.append(
    f"HIGHEST PRIORITY: {top['PU_zone']} in {top['PU_borough']} "
    f"has priority score {top['avg_priority_score']:.1f}/100 "
    f"with avg speed {top['avg_speed']:.1f} mph and {top['total_trips']:,.0f} trips."
)

insights.append(
    f"CONGESTION IMPACT: Speed drops {drop:.1f}% during peak hours "
    f"({pk_speed:.1f} mph vs {opk_speed:.1f} mph off-peak). "
    f"Congestion rate is {pk['is_congested_trip'].mean()*100:.1f}% during peak."
)

insights.append(
    f"DEMAND PATTERN: Busiest hours are "
    + ", ".join([f"{h:02d}:00" for h in hourly.head(3).index])
    + f". Weekday avg {wd//23:,} trips/day vs Weekend {we//8:,} trips/day."
)

midtown = zs[zs['PU_zone'].str.contains('Midtown', na=False)]
if len(midtown) > 0:
    mt = midtown.iloc[0]
    insights.append(
        f"MIDTOWN FOCUS: {mt['PU_zone']} has {mt['total_trips']:,.0f} total trips "
        f"with {mt['high_hours']:.0f} high-priority hours. "
        f"This is the #1 candidate for transport optimization."
    )

with open('../data/processed/insights_report.txt', 'w') as f:
    for i, insight in enumerate(insights, 1):
        f.write(f"Insight {i}: {insight}\n\n")

print("Insights saved to data/processed/insights_report.txt\n")
for i, insight in enumerate(insights, 1):
    print(f"Insight {i}: {insight}\n")

Insights saved to data/processed/insights_report.txt

Insight 1: HIGHEST PRIORITY: Brownsville in Brooklyn has priority score 38.4/100 with avg speed 7.6 mph and 141 trips.

Insight 2: CONGESTION IMPACT: Speed drops 4.6% during peak hours (9.4 mph vs 9.9 mph off-peak). Congestion rate is 64.5% during peak.

Insight 3: DEMAND PATTERN: Busiest hours are 18:00, 17:00, 19:00. Weekday avg 75,191 trips/day vs Weekend 76,118 trips/day.

Insight 4: MIDTOWN FOCUS: Midtown Center has 125,288 total trips with 6 high-priority hours. This is the #1 candidate for transport optimization.

